# CS-781 Phase 1 — Architecture 3: Encoder–Decoder Transformer (~79M params, 400M-token plan) — **v3 FINAL**

True encoder–decoder Transformer: 9-layer bidirectional encoder, 9-layer causal decoder with cross-attention,
d_model 512, 8 heads, FFN 2048, tied embedding / output projection, vocab 24,576 (~78.8M parameters — unchanged from v1).
Corpus budget: 400M token IDs (SC 144M, HC 144M, Tribunals 72M, Acts 40M). Data pipeline, tokenizer and evaluation code are the same as v1,
so a previously built `data/` directory can be reused via `REUSE_DATA_FROM`.

## What changed in v2 (parameter count and data format untouched)

| # | Change | Why |
|---|--------|-----|
| 1 | Token embeddings × √d_model before adding sinusoidal positions | v1 added unit-scale positions to 0.02-std embeddings (positions ≈ 35× larger than tokens at init) |
| 2 | **Document-boundary masking** (`MASK_DOC_BOUNDARIES`) | Packed blocks cross `</s>`; v1 trained the decoder to "continue" into an unrelated next document and let the encoder see an older document |
| 3 | **Decoder priming** (`DEC_PREFIX_TOKENS=64`) | The decoder first reads `<s>` + the last 64 prompt tokens (no loss on them), so the first generated words are locally coherent instead of relying only on cross-attention to find the prompt's end |
| 4 | `<s>` end-of-prompt marker on the encoder (`ENC_END_MARKER`) | Prompt length is random, so cross-attention had no explicit "end of prompt" signal |
| 5 | Validation loss: fixed prompt-length set, token-weighted, same-document tokens only | v1 used one fixed split (P95) — validation did not match the training mixture. **v2 loss values are not comparable to v1** |
| 6 | Cell 9b **self-test** | Verifies padding invariance, KV-cache == full decoding, causality, cross-attention use, label alignment |
| 7 | `build_model()`; Cell 11 starts from fresh weights | Removes the fragile "rerun Cell 9 after the pilot" step |
| 8 | No weight decay on the tied embedding; decoder init uses 3 residual branches | Standard practice |
| 9 | Generation: whitespace tokens exempt from repetition penalty; last word must be complete; per-call decoding args; Cell 13b decoding sweep | This BPE emits every space as its own token, so the penalty hit them constantly; overlap metrics usually prefer low temperature |

## Workflow

1. Run Cells 1–9b (Cell 9b must print `SELF-TEST PASSED`).
2. Cell 10 in `pilot` mode → Cell 11 for ~100–200 optimizer steps; check loss, tokens/s, ETA.
3. Stop, set `RUN_MODE = "full"` in Cell 10, rerun Cell 10 and Cell 11.
4. After training: Cell 12 (local generation), Cell 13 (metrics), Cell 13b (decoding sweep) → set the best decoding values in Cell 2 → Cell 14 (official ZIP).

### v3 review notes

This version keeps the **same Phase-1 objective**: encode a legal prompt and autoregressively generate its continuation.  
The 400M-token corpus budget and ~79M parameter target are unchanged.

Additional safeguards retained/adopted after notebook review:

- scaled token embeddings with sinusoidal positions;
- document-boundary-aware seq2seq examples;
- 64-token decoder prompt priming, used consistently in training and inference;
- encoder end marker;
- KV-cache / causality / cross-attention / label-alignment self-test;
- token-weighted, deterministic validation;
- whitespace tokens exempt from repetition penalty;
- decoding sweep, with **BERTScore used to select among the best overlap-metric candidates** because BERTScore F1 is the Phase-1 primary metric;
- strict exact-128-word and ZIP validation before submission.


In [1]:
# Cell 1 — Install dependencies
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ijson",
    "huggingface_hub",
    "transformers==4.48.3",
    "bert-score==0.3.13",
])

print("Dependencies ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 88.7 MB/s eta 0:00:00
Dependencies ready.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Cell 2 — Imports + encoder–decoder full-run configuration
import os
if os.path.isdir("/kaggle/temp"):
    os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")

import re, gc, json, math, time, random, zipfile, heapq, shutil
from pathlib import Path
from collections import Counter

import ijson
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, login

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded.")
except Exception as e:
    print("HF secret unavailable; public access will be attempted.", repr(e))

if HF_TOKEN:
    login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILES = {
    "sc":   "Phase_1_Pretraining/Train/train_supreme_court.json",
    "hc":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tr":   "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}

# ----------------------------- output paths
ROOT = Path("/kaggle/working/cs781_encdec_80m")
DATA_DIR = ROOT / "data"
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_PATH = DATA_DIR / "tokenizer.json"
TRAIN_BIN = DATA_DIR / "train_all.bin"
VAL_BIN = DATA_DIR / "val_all.bin"
META_PATH = DATA_DIR / "meta.json"
VAL_GEN_PATH = DATA_DIR / "val_gen.json"
VAL_REF_PATH = DATA_DIR / "val_ref.json"

# Optional: point to a prior compatible tokenized data directory.
# Leave None to build this model's fresh 400M corpus.
REUSE_DATA_FROM = None
if REUSE_DATA_FROM:
    reuse_dir = Path(REUSE_DATA_FROM)
    for _name in [
        "tokenizer.json", "census.json", "meta.json",
        "train_all.bin", "val_all.bin", "val_gen.json", "val_ref.json"
    ]:
        _src, _dst = reuse_dir / _name, DATA_DIR / _name
        if _src.exists() and not _dst.exists():
            shutil.copy2(_src, _dst)
            print("Reused:", _name)

# Run-specific checkpoint paths are configured in Cell 10.
RUN_NAME = None
CKPT_DIR = None
LATEST_CKPT = None
BEST_CKPT = None
FINAL_CKPT = None
OUTSIDE_LATEST_CKPT = None
OUTSIDE_BEST_CKPT = None

HF_CHECKPOINT_REPO = None
HF_UPLOAD_EVERY = 2000
HF_SQUASH_EVERY_N_UPLOADS = 4

LOCAL_PRED = ROOT / "val_pred.json"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# ----------------------------- tokenizer
VOCAB_SIZE = 24_576
TOKENIZER_CHAR_BUDGET = {
    "sc":   16_000_000,
    "hc":   16_000_000,
    "tr":   10_000_000,
    "acts":  6_000_000,
}
TOKENIZER_MIN_COUNT = 2

# ----------------------------- 400M pretraining corpus
TRAIN_TOKEN_BUDGET = {
    "sc":   144_000_000,
    "hc":   144_000_000,
    "tr":    72_000_000,
    "acts":  40_000_000,
}
TARGET_TOTAL_TOKENS = sum(TRAIN_TOKEN_BUDGET.values())

VAL_TOKENS_PER_SOURCE = 500_000
VAL_GEN_PER_SOURCE = 40
VAL_PERMILLE = 5
DELETE_SOURCE_AFTER_STAGE = True
ENCODE_PROCESSES = min(4, os.cpu_count() or 2)

# ----------------------------- context / model
SEQ_LEN = None
MAX_POSITIONS = 2048               # preprocessing safety ceiling
MAX_ENCODER_POSITIONS = 2048
MAX_DECODER_POSITIONS = 1536

# ~78.8M trainable parameters.
D_MODEL = 512
N_ENCODER_LAYERS = 9
N_DECODER_LAYERS = 9
N_HEADS = 8
FFN_DIM = 2048
DROPOUT = 0.0

# ----------------------------- v2 design switches (see Cell 9b self-test)
DEC_PREFIX_TOKENS = 64        # last-K prompt tokens are also fed to the decoder as primed, loss-free context (0 = off)
ENC_END_MARKER = True         # append <s> to the encoder input so cross-attention can find the end of the prompt
MASK_DOC_BOUNDARIES = True    # never train across a </s> document boundary (encoder prefix or continuation)
MIN_CTX_TOKENS = 32           # rows with less same-document prompt than this are skipped in the loss

PROMPT_P10_TOKENS = None
PROMPT_P95_TOKENS = None
EXPECTED_CONT_TOKENS = None
PROMPT_SPLIT_MIN = None
PROMPT_SPLIT_MAX = None
PROMPT_KEEP_TOKENS = None

# ----------------------------- optimization
MICRO_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
MIN_LR_RATIO = 0.10
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.10
GRAD_CLIP = 1.0
LR_SCHEDULE = "wsd"
WSD_DECAY_FRAC = 0.20

TRAIN_BLOCK_LIMIT = None
LOG_EVERY = 50
VAL_EVERY = 1000
SAVE_EVERY = 500
VAL_EVAL_BLOCKS = 256

# ----------------------------- generation
TARGET_GENERATION_WORDS = 128
GEN_TEMPERATURE = 0.8
GEN_TOP_P = 0.95
GEN_TOP_K = 0
GEN_REP_PENALTY = 1.03

# Encoder and decoder have separate positional spaces. The decoder therefore
# has plenty of room even when the encoder consumes a long 512-word prompt.
GEN_MAX_NEW_TOKENS = 1280
GEN_BATCH_SIZE = 16
OFFICIAL_PARTIAL_EVERY = 4

FAST_EVAL_NO_BERT = False
BERTSCORE_BATCH_SIZE = 8
GENERATION_SEED = SEED + 2026   # reproducible sampling start for a fresh official run
RUN_OFFICIAL_TEST = False

print("Output:", ROOT)
print("Target train tokens:", f"{TARGET_TOTAL_TOKENS/1e6:.0f}M")


Device: cuda
GPU: Tesla T4
HF token loaded.
Output: /kaggle/working/cs781_encdec_80m
Target train tokens: 400M


In [3]:
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

Tokenizer class ready.


In [4]:
# Cell 4 — Streaming source utilities + robust test discovery
def iter_docs(path):
    """Yield document text from the official top-level JSON array."""
    with open(path, "rb") as f:
        for obj in ijson.items(f, "item"):
            text = obj.get("text", "")
            if isinstance(text, str) and text.strip():
                yield text

def fetch_source(key):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=SOURCE_FILES[key],
        repo_type="dataset",
        token=HF_TOKEN,
    )

def find_test_file():
    files = HfApi().list_repo_files(
        HF_DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    candidates = [
        f for f in files
        if f.startswith("Phase_1_Pretraining/Test") and f.endswith(".json")
    ]
    if not candidates:
        raise RuntimeError("Could not discover Phase-1 test JSON.")
    chosen = candidates[0]
    print("Official test file:", chosen)
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=chosen,
        repo_type="dataset",
        token=HF_TOKEN,
    )

def maybe_delete_download(path):
    if not DELETE_SOURCE_AFTER_STAGE:
        return
    try:
        real = os.path.realpath(path)
        if os.path.exists(real):
            os.remove(real)
            print("Deleted cached source to save disk:", real)
    except Exception as e:
        print("Could not delete source cache:", repr(e))

def source_census(key, sample_docs=1000):
    path = fetch_source(key)
    size = os.path.getsize(path)
    n = chars = words = 0
    lens = []
    for doc in iter_docs(path):
        n += 1
        chars += len(doc)
        w = len(doc.split())
        words += w
        lens.append(w)
        if n >= sample_docs:
            break
    avg_chars = chars / max(n,1)
    # JSON overhead correction; only used to choose approximate uniform sampling probability.
    est_docs = int(size / max(1.0, avg_chars * 1.03))
    est_chars = avg_chars * est_docs
    return {
        "file_bytes":size,
        "sample_docs":n,
        "avg_chars":avg_chars,
        "avg_words":words/max(n,1),
        "est_docs":est_docs,
        "est_chars":est_chars,
        "p10_p50_p90_words":[int(np.percentile(lens,q)) for q in (10,50,90)] if lens else [],
    }

print("Streaming utilities ready.")

Streaming utilities ready.


In [5]:
# Cell 5 — Census + train BPE on a uniform sample from ALL FOUR sources
CENSUS_PATH = DATA_DIR / "census.json"

if CENSUS_PATH.exists():
    census = json.load(open(CENSUS_PATH))
    print("Loaded existing census.")
else:
    census = {}
    for key in SOURCE_FILES:
        t0=time.time()
        census[key]=source_census(key, sample_docs=1000)
        print(
            key,
            f"{census[key]['file_bytes']/1e9:.2f} GB",
            "avg_chars", f"{census[key]['avg_chars']:,.0f}",
            "est_docs", f"{census[key]['est_docs']:,}",
            "words p10/50/90", census[key]["p10_p50_p90_words"],
            f"({time.time()-t0:.0f}s)"
        )
        # Keep downloaded file for tokenizer stage in this same session.
    json.dump(census, open(CENSUS_PATH,"w"), indent=2)

if TOKENIZER_PATH.exists():
    tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
    print("Loaded existing tokenizer:", len(tokenizer))
else:
    rng = random.Random(SEED)

    def tokenizer_text_stream():
        for key, char_budget in TOKENIZER_CHAR_BUDGET.items():
            path = fetch_source(key)
            est_chars = max(1.0, census[key]["est_chars"])
            # Slight oversampling makes it unlikely we finish far below target.
            q = min(1.0, 1.05 * char_budget / est_chars)
            got = 0
            seen = 0
            print(f"[{key}] tokenizer sample probability={q:.6f}")
            for doc in iter_docs(path):
                seen += 1
                if rng.random() < q:
                    # Cap unusually huge documents so one case cannot dominate merge learning.
                    d = doc[:30000]
                    got += len(d)
                    yield d
                    if got >= char_budget:
                        break
            print(f"[{key}] tokenizer chars={got/1e6:.1f}M | docs scanned={seen:,}", flush=True)
            # Do NOT delete here: Cell 7 re-reads this file (all four are already on disk after the census),
            # so keeping it avoids downloading every source twice. Cell 7 deletes each file when finished.

    tokenizer = LegalBPE(VOCAB_SIZE)
    t0=time.time()
    tokenizer.fit(tokenizer_text_stream(), min_count=TOKENIZER_MIN_COUNT)
    tokenizer.save_json(TOKENIZER_PATH)
    print(f"Tokenizer saved to {TOKENIZER_PATH}")
    print(f"Tokenizer training elapsed: {(time.time()-t0)/60:.1f} min")

# Exact byte-level round-trip sanity.
probe = "IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
roundtrip = tokenizer.decode(tokenizer.encode(probe))
print("Round-trip exact:", roundtrip == probe)
print("Vocab:", len(tokenizer))

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

sc 1.32 GB avg_chars 36,184 est_docs 35,486 words p10/50/90 [1580, 3869, 11757] (5s)


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

hc 10.67 GB avg_chars 2,648 est_docs 3,913,049 words p10/50/90 [68, 206, 960] (42s)


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

tr 6.48 GB avg_chars 27,783 est_docs 226,580 words p10/50/90 [238, 3432, 9831] (34s)


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

acts 0.73 GB avg_chars 44,418 est_docs 15,929 words p10/50/90 [407, 2827, 14848] (8s)
[sc] tokenizer sample probability=0.013084
  BPE corpus read: 10M chars | unique units=78,714
[sc] tokenizer chars=11.1M | docs scanned=55,528
[hc] tokenizer sample probability=0.001621
  BPE corpus read: 20M chars | unique units=146,851
[hc] tokenizer chars=13.8M | docs scanned=2,536,738
[tr] tokenizer sample probability=0.001668
  BPE corpus read: 30M chars | unique units=216,937
[tr] tokenizer chars=9.0M | docs scanned=869,132
[acts] tokenizer sample probability=0.008904
[acts] tokenizer chars=2.9M | docs scanned=18,822
BPE training units after min_count=2: 103,942
Counting time: 1.4 min
merge 1/24,316 | freq=760,292 | vocab=261
merge 1,000/24,316 | freq=2,684 | vocab=1,260
merge 2,000/24,316 | freq=1,112 | vocab=2,260
merge 3,000/24,316 | freq=635 | vocab=3,260
merge 4,000/24,316 | freq=415 | vocab=4,260
merge 5,000/24,316 | freq=297 | vocab=5,260
merge 6,000/24,316 | freq=225 | vocab=6,260
merge 

In [6]:
# Cell 6 — Measure official 512-word prompts and choose the encoder/decoder training block
test_path = find_test_file()
test_data_for_stats = json.load(open(test_path, encoding="utf-8"))

prompt_token_lens = []
prompt_word_lens = []

for rec in test_data_for_stats.values():
    text = rec["text"]
    prompt_token_lens.append(len(tokenizer.encode(text)))
    prompt_word_lens.append(len(text.split()))

tpw = sum(prompt_token_lens) / max(1, sum(prompt_word_lens))

PROMPT_P10_TOKENS = int(np.percentile(prompt_token_lens, 10))
PROMPT_P95_TOKENS = int(np.percentile(prompt_token_lens, 95))

raw_cont_budget = int(math.ceil(128 * tpw * 1.35))
EXPECTED_CONT_TOKENS = int(math.ceil(raw_cont_budget / 64) * 64)

# Keep the same 1536-ish raw training block design as Architecture 2.
SEQ_LEN = int(math.ceil((PROMPT_P95_TOKENS + EXPECTED_CONT_TOKENS) / 128) * 128)

if SEQ_LEN > MAX_POSITIONS:
    raise RuntimeError(
        f"Measured SEQ_LEN={SEQ_LEN} exceeds MAX_POSITIONS={MAX_POSITIONS}."
    )

PROMPT_SPLIT_MIN = max(
    64,
    min(PROMPT_P10_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)
)
PROMPT_SPLIT_MAX = max(
    PROMPT_SPLIT_MIN,
    min(PROMPT_P95_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)
)
PROMPT_KEEP_TOKENS = min(PROMPT_P95_TOKENS, MAX_ENCODER_POSITIONS)

print("Official prompts:", len(prompt_token_lens))
print(
    "BPE tokens min/mean/p10/p95/max:",
    min(prompt_token_lens),
    f"{np.mean(prompt_token_lens):.0f}",
    PROMPT_P10_TOKENS,
    PROMPT_P95_TOKENS,
    max(prompt_token_lens),
)
print("Tokens/word:", round(tpw, 3))
print("Expected 128-word continuation budget:", EXPECTED_CONT_TOKENS, "BPE tokens")
print("CHOSEN SEQ_LEN:", SEQ_LEN)
print("Training encoder-prefix range:", PROMPT_SPLIT_MIN, "to", PROMPT_SPLIT_MAX)
print("Inference prompt keep:", PROMPT_KEEP_TOKENS)
print("Encoder max positions:", MAX_ENCODER_POSITIONS)
print("Decoder max positions:", MAX_DECODER_POSITIONS)

assert PROMPT_SPLIT_MAX < SEQ_LEN
assert PROMPT_KEEP_TOKENS <= MAX_ENCODER_POSITIONS


Official test file: Phase_1_Pretraining/Test/test.json


test.json: 0.00B [00:00, ?B/s]

Official prompts: 1000
BPE tokens min/mean/p10/p95/max: 1030 1082 1050 1139 1367
Tokens/word: 2.114
Expected 128-word continuation budget: 384 BPE tokens
CHOSEN SEQ_LEN: 1536
Training encoder-prefix range: 1050 to 1139
Inference prompt keep: 1139
Encoder max positions: 2048
Decoder max positions: 1536


In [7]:
# Cell 7 — Stream/tokenize ALL FOUR sources to uint16 train/val binaries (400M train target)
# This is the CPU-heavy preprocessing stage. It is resumable at the FILE level:
# if meta.json + both bins already exist, this entire stage is skipped.

_WORKER_TOKENIZER = None
_WORKER_EOS = None

def _encode_worker(text):
    ids = _WORKER_TOKENIZER.encode(text)
    ids.append(_WORKER_EOS)
    return np.asarray(ids, dtype=np.uint16)

def make_pool():
    global _WORKER_TOKENIZER, _WORKER_EOS
    _WORKER_TOKENIZER = tokenizer
    _WORKER_EOS = tokenizer.eos_token_id

    if ENCODE_PROCESSES <= 1:
        return None

    import multiprocessing as mp
    try:
        ctx = mp.get_context("fork")
        return ctx.Pool(ENCODE_PROCESSES)
    except Exception as e:
        print("Multiprocessing unavailable; falling back to single process:", repr(e))
        return None

def encode_many(texts, pool):
    if pool is None:
        return [_encode_worker(x) for x in texts]
    return list(pool.imap(_encode_worker, texts, chunksize=8))

if META_PATH.exists() and TRAIN_BIN.exists() and VAL_BIN.exists() and VAL_GEN_PATH.exists() and VAL_REF_PATH.exists():
    meta = json.load(open(META_PATH))
    print("Existing tokenized corpus found; skipping preprocessing.")
    print(json.dumps(meta, indent=2)[:2500])
else:
    # Start fresh so partial old bins cannot be mistaken for complete data.
    for p in [TRAIN_BIN, VAL_BIN, VAL_GEN_PATH, VAL_REF_PATH, META_PATH]:
        if p.exists():
            p.unlink()

    f_train = open(TRAIN_BIN, "wb")
    f_val = open(VAL_BIN, "wb")
    pool = make_pool()

    meta = {
        "vocab_size":len(tokenizer),
        "seq_len":SEQ_LEN,
        "prompt_p10_tokens":PROMPT_P10_TOKENS,
        "prompt_p95_tokens":PROMPT_P95_TOKENS,
        "expected_cont_tokens":EXPECTED_CONT_TOKENS,
        "prompt_split_min":PROMPT_SPLIT_MIN,
        "prompt_split_max":PROMPT_SPLIT_MAX,
        "sources":{},
        "target_total_tokens":TARGET_TOTAL_TOKENS,
    }
    val_gen = {}
    val_ref = {}

    rng = random.Random(SEED + 99)

    try:
        for key, target in TRAIN_TOKEN_BUDGET.items():
            path = fetch_source(key)

            # Estimate tokens/char from the first ~200 documents.
            probe_docs = []
            probe_chars = 0
            for doc in iter_docs(path):
                probe_docs.append(doc)
                probe_chars += len(doc)
                if len(probe_docs) >= 200:
                    break

            probe_encoded = encode_many(probe_docs, pool)
            probe_tokens = sum(len(x) for x in probe_encoded)
            tokens_per_char = probe_tokens / max(1, probe_chars)
            est_tokens = census[key]["est_chars"] * tokens_per_char

            # Oversample ~8%, then stop exactly when the requested token budget is reached.
            q_train = min(1.0, 1.08 * target / max(1.0, est_tokens))
            q_val = min(
                1.0,
                1.20 * VAL_TOKENS_PER_SOURCE /
                max(1.0, est_tokens * VAL_PERMILLE / 1000.0)
            )

            print(
                f"\n[{key}] est_tokens≈{est_tokens/1e6:.0f}M | "
                f"target={target/1e6:.0f}M | q_train={q_train:.5f} | q_val={q_val:.5f}",
                flush=True
            )

            n_train = 0
            n_val = 0
            docs_seen = 0
            gen_found = 0
            t0 = time.time()

            batch_texts = []
            batch_is_val = []

            def flush_batch():
                global n_train, n_val, batch_texts, batch_is_val
                if not batch_texts:
                    return

                arrays = encode_many(batch_texts, pool)

                for arr, is_val in zip(arrays, batch_is_val):
                    if is_val:
                        if n_val >= VAL_TOKENS_PER_SOURCE:
                            continue
                        remaining = VAL_TOKENS_PER_SOURCE - n_val
                        a = arr[:remaining]
                        a.tofile(f_val)
                        n_val += len(a)
                    else:
                        if n_train >= target:
                            continue
                        remaining = target - n_train
                        a = arr[:remaining]
                        a.tofile(f_train)
                        n_train += len(a)

                batch_texts = []
                batch_is_val = []

            for idx, doc in enumerate(iter_docs(path)):
                docs_seen += 1
                if len(doc) < 200:
                    continue

                # Deterministic document-level heldout split.
                is_val_doc = ((idx * 2654435761) % 1000) < VAL_PERMILLE

                if is_val_doc:
                    if n_val >= VAL_TOKENS_PER_SOURCE:
                        continue
                    if rng.random() > q_val:
                        continue

                    words = doc.split()
                    if gen_found < VAL_GEN_PER_SOURCE and len(words) >= 640:
                        st = rng.randint(0, len(words)-640)
                        exid = f"{key}_{gen_found:02d}"
                        val_gen[exid] = {"text":" ".join(words[st:st+512])}
                        val_ref[exid] = {"text":" ".join(words[st+512:st+640])}
                        gen_found += 1

                    batch_texts.append(doc)
                    batch_is_val.append(True)

                else:
                    if n_train >= target:
                        # Keep scanning only until validation is also sufficiently filled.
                        if n_val >= int(0.95 * VAL_TOKENS_PER_SOURCE):
                            break
                        continue
                    if rng.random() > q_train:
                        continue
                    batch_texts.append(doc)
                    batch_is_val.append(False)

                if len(batch_texts) >= 64:
                    flush_batch()

                    if n_train and n_train % 25_000_000 < 100_000:
                        print(
                            f"  [{key}] train={n_train/1e6:.1f}M "
                            f"val={n_val/1e6:.2f}M docs_seen={docs_seen:,} "
                            f"elapsed={(time.time()-t0)/60:.1f}m",
                            flush=True
                        )

            flush_batch()

            meta["sources"][key] = {
                "train_tokens":n_train,
                "val_tokens":n_val,
                "docs_seen":docs_seen,
                "tokens_per_char_probe":tokens_per_char,
                "q_train":q_train,
            }

            print(
                f"[{key}] DONE train={n_train/1e6:.1f}M "
                f"val={n_val/1e6:.2f}M local_gen={gen_found} "
                f"time={(time.time()-t0)/60:.1f}m"
            )

            maybe_delete_download(path)

    finally:
        if pool is not None:
            pool.close()
            pool.join()
        f_train.close()
        f_val.close()

    json.dump(val_gen, open(VAL_GEN_PATH,"w"), ensure_ascii=False, indent=2)
    json.dump(val_ref, open(VAL_REF_PATH,"w"), ensure_ascii=False, indent=2)

    meta["actual_train_tokens"] = os.path.getsize(TRAIN_BIN)//2
    meta["actual_val_tokens"] = os.path.getsize(VAL_BIN)//2
    json.dump(meta, open(META_PATH,"w"), indent=2)

    print("\nPreprocessing complete.")
    print("train tokens:", f"{meta['actual_train_tokens']/1e6:.1f}M")
    print("val tokens:", f"{meta['actual_val_tokens']/1e6:.2f}M")
    print("binary train size:", f"{os.path.getsize(TRAIN_BIN)/1e9:.2f} GB")
    print("local generation examples:", len(val_gen))


[sc] est_tokens≈474M | target=144M | q_train=0.32836 | q_val=0.25336
  [sc] train=25.1M val=0.10M docs_seen=8,623 elapsed=0.2m
  [sc] train=50.0M val=0.22M docs_seen=17,251 elapsed=0.4m
[sc] DONE train=144.0M val=0.50M local_gen=40 time=1.2m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/96d8e4d106292e7d5b6c4cb81e125f7e8cdee50b66302a373f547178b5364923

[hc] est_tokens≈3652M | target=144M | q_train=0.04258 | q_val=0.03286
  [hc] train=0.0M val=0.00M docs_seen=1,523 elapsed=0.0m
  [hc] train=25.0M val=0.13M docs_seen=585,195 elapsed=0.4m
  [hc] train=75.1M val=0.26M docs_seen=1,102,593 elapsed=1.0m
  [hc] train=100.0M val=0.32M docs_seen=1,604,216 elapsed=1.4m
  [hc] train=100.1M val=0.32M docs_seen=1,605,911 elapsed=1.4m
[hc] DONE train=144.0M val=0.46M local_gen=40 time=2.0m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/59b3d4df6b492dd03d89197aa634b583

In [8]:
# Cell 8 — Memory-mapped block dataset sanity
meta = json.load(open(META_PATH))
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_mm = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

BLOCK_TOKENS = SEQ_LEN + 1
N_TRAIN_BLOCKS = len(train_mm) // BLOCK_TOKENS
N_VAL_BLOCKS = len(val_mm) // BLOCK_TOKENS

print("Train token IDs:", f"{len(train_mm):,}")
print("Val token IDs:", f"{len(val_mm):,}")
print("Train blocks:", f"{N_TRAIN_BLOCKS:,}")
print("Val blocks:", f"{N_VAL_BLOCKS:,}")

assert N_TRAIN_BLOCKS > 0 and N_VAL_BLOCKS > 0
assert len(tokenizer) < 65536

sample = np.asarray(train_mm[:BLOCK_TOKENS], dtype=np.int64)
print("Sample block shape:", sample.shape)
print("Decoded start:", repr(tokenizer.decode(sample[:80])[:300]))

# ---- 400M corpus sanity
print("Requested training-token budget:", f"{TARGET_TOTAL_TOKENS:,}")
assert len(train_mm) >= TARGET_TOTAL_TOKENS, (
    f"Training binary is too small: {len(train_mm):,} < requested {TARGET_TOTAL_TOKENS:,}"
)
print("✅ 400M corpus sanity passed")


Train token IDs: 400,000,000
Val token IDs: 1,959,805
Train blocks: 260,247
Val blocks: 1,275
Sample block shape: (1537,)
Decoded start: 'JUDGMENT\n\nK.C. Das Gupta, J.\n\n1. The only question that arises for consideration in this appeal by the State Bank of India is as regards the effect of Para 292 of the Bank Award, as modified, on the respondent, '
Requested training-token budget: 400,000,000
✅ 400M corpus sanity passed


In [9]:
# Cell 9 — ~79M Encoder–Decoder Transformer (SDPA, cross-attention, decode KV cache)
# v2 changes vs v1:
#   * token embeddings are scaled by sqrt(d_model) before the sinusoidal positions are added
#     (v1 added unit-scale positions to 0.02-std embeddings => positions drowned the tokens)
#   * encoder positions are derived from the padding mask (so padded / doc-masked rows get positions 0..n-1)
#   * decoder self-attention handles "cache + several new tokens" correctly (needed for prompt priming)
#   * residual-branch init uses the true branch count (3 per decoder layer)
#   * build_model() so Cell 11 always starts from fresh weights (no more "rerun Cell 9" step)
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class EncoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, key_valid=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        attn_mask = None
        if key_valid is not None:
            attn_mask = key_valid[:, None, None, :]  # True = valid encoder key

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y)


class DecoderSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        mask, is_causal = None, True
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
            if T == 1:
                is_causal = False                      # the new token may see every cached key
            else:
                S = k.size(2)
                P = S - T                              # query i sits at global position P+i
                mask = torch.ones(T, S, device=x.device).tril(diagonal=P).bool()
                is_causal = False

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.kv_proj = nn.Linear(d_model, 2 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, memory, memory_valid=None, cached_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if cached_kv is None:
            S = memory.size(1)
            kv = self.kv_proj(memory).view(B, S, 2, self.n_heads, self.head_dim)
            k, v = kv.unbind(dim=2)
            k, v = k.transpose(1, 2), v.transpose(1, 2)
        else:
            k, v = cached_kv

        attn_mask = None
        if memory_valid is not None:
            attn_mask = memory_valid[:, None, None, :]

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = EncoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_valid=None):
        x = x + self.drop(self.attn(self.ln1(x), key_valid=key_valid))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = DecoderSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_heads, dropout)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.drop = nn.Dropout(dropout)

    def forward(
        self, x, memory, memory_valid=None,
        past_self_kv=None, cached_cross_kv=None, use_cache=False
    ):
        a, new_self = self.self_attn(
            self.ln1(x), past_kv=past_self_kv, use_cache=use_cache
        )
        x = x + self.drop(a)

        c, new_cross = self.cross_attn(
            self.ln2(x),
            memory,
            memory_valid=memory_valid,
            cached_kv=cached_cross_kv,
            use_cache=use_cache,
        )
        x = x + self.drop(c)
        x = x + self.ff(self.ln3(x))
        return x, new_self, new_cross


class EncoderDecoderLM(nn.Module):
    def __init__(
        self, vocab_size, d_model, n_encoder_layers, n_decoder_layers,
        n_heads, ffn_dim, dropout, max_encoder_positions, max_decoder_positions
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.emb_scale = math.sqrt(d_model)
        self.max_encoder_positions = max_encoder_positions
        self.max_decoder_positions = max_decoder_positions

        # Shared embedding across encoder input, decoder input and output projection.
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.enc_pos = SinusoidalPositionalEncoding(d_model, max_encoder_positions)
        self.dec_pos = SinusoidalPositionalEncoding(d_model, max_decoder_positions)
        self.drop = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([
            EncoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_encoder_layers)
        ])
        self.decoder = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_decoder_layers)
        ])
        self.enc_ln = nn.LayerNorm(d_model)
        self.dec_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-2 style depth scaling: std = 0.02 / sqrt(#residual branches)
        enc_scaled = 0.02 / math.sqrt(2 * n_encoder_layers)   # attn + ffn
        dec_scaled = 0.02 / math.sqrt(3 * n_decoder_layers)   # self-attn + cross-attn + ffn
        for block in self.encoder:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=enc_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=enc_scaled)
        for block in self.decoder:
            nn.init.normal_(block.self_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.cross_attn.out_proj.weight, mean=0.0, std=dec_scaled)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=dec_scaled)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def encode(self, input_ids, key_valid=None, position_ids=None):
        B, S = input_ids.shape
        if S > self.max_encoder_positions:
            raise ValueError(f"encoder length {S} > {self.max_encoder_positions}")

        if position_ids is None:
            if key_valid is None:
                position_ids = torch.arange(S, device=input_ids.device)[None, :].expand(B, S)
            else:
                # valid tokens always get positions 0..n-1, wherever the padding / masked tokens sit
                position_ids = (key_valid.long().cumsum(1) - 1).clamp(min=0)

        x = self.token_emb(input_ids) * self.emb_scale \
            + self.enc_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)
        for block in self.encoder:
            x = block(x, key_valid=key_valid)
        return self.enc_ln(x)

    def decode_hidden(
        self, decoder_ids, memory, memory_valid=None,
        past_self_key_values=None, cross_key_values=None,
        use_cache=False, position_ids=None
    ):
        B, T = decoder_ids.shape
        past_len = 0
        if past_self_key_values is not None:
            past_len = past_self_key_values[0][0].size(2)

        if past_len + T > self.max_decoder_positions:
            raise ValueError(
                f"decoder length {past_len + T} > {self.max_decoder_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=decoder_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(decoder_ids) * self.emb_scale \
            + self.dec_pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_self, new_cross = ([] if use_cache else None), ([] if use_cache else None)

        for i, block in enumerate(self.decoder):
            ps = None if past_self_key_values is None else past_self_key_values[i]
            ck = None if cross_key_values is None else cross_key_values[i]
            x, s_kv, c_kv = block(
                x, memory, memory_valid=memory_valid,
                past_self_kv=ps, cached_cross_kv=ck, use_cache=use_cache
            )
            if use_cache:
                new_self.append(s_kv)
                new_cross.append(c_kv)

        return self.dec_ln(x), new_self, new_cross

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_model():
    return EncoderDecoderLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_encoder_layers=N_ENCODER_LAYERS,
        n_decoder_layers=N_DECODER_LAYERS,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_encoder_positions=MAX_ENCODER_POSITIONS,
        max_decoder_positions=MAX_DECODER_POSITIONS,
    ).to(DEVICE)

# ---- instantiate
model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")
print("Architecture: 9-layer bidirectional encoder + 9-layer causal cross-attention decoder")
assert 75_000_000 <= n_params <= 85_000_000, n_params


Model parameters: 78.79M
Architecture: 9-layer bidirectional encoder + 9-layer causal cross-attention decoder


In [10]:
# Cell 9b — Seq2seq batch builder + loss + SELF-TEST (run once; takes seconds)
# The self-test proves the things that silently ruin an encoder–decoder run:
#   (a) padded encoder == un-padded encoder, (b) cached decoding == full teacher-forced decoding,
#   (c) the decoder is causal, (d) the decoder really reads the encoder,
#   (e) document-boundary masking / prompt priming produce correctly aligned labels.
IGNORE = -100

def make_seq2seq_batch(batch, split):
    """
    batch : (B, SEQ_LEN+1) token ids sliced from the packed corpus stream.
    split : number of tokens that form the "prompt" (random in training, like the official 512-word prompts).

    Encoder input : prompt tokens (+ <s> end marker). Tokens that belong to an *older* document
                    (everything up to the last </s> inside the prompt) are masked out.
    Decoder input : <s> + last K prompt tokens (primed context, no loss) + continuation[:-1]
    Labels        : IGNORE for the K primed positions; continuation tokens afterwards;
                    IGNORE at/after the first </s> in the continuation (that would be an unrelated next document).
    Returns (enc_ids, enc_valid, dec_in, labels, skip) with skip = K = number of primed decoder positions.
    """
    split = int(split)
    B, dev = batch.size(0), batch.device
    EOS, BOS = tokenizer.eos_token_id, tokenizer.bos_token_id

    prefix = batch[:, :split]
    targets = batch[:, split:]

    # ---- encoder side
    idx = torch.arange(split, device=dev)[None, :].expand(B, split)
    if MASK_DOC_BOUNDARIES:
        last_eos = torch.where(prefix == EOS, idx, torch.full_like(idx, -1)).max(dim=1).values
        enc_valid = idx > last_eos[:, None]
    else:
        enc_valid = torch.ones(B, split, dtype=torch.bool, device=dev)

    short_ctx = enc_valid.sum(dim=1) < MIN_CTX_TOKENS      # almost no same-document prompt left
    enc_valid = enc_valid | short_ctx[:, None]              # keep >=1 valid key (loss of these rows is masked below)

    enc_ids = prefix
    if ENC_END_MARKER:
        enc_ids = torch.cat([prefix, torch.full((B, 1), BOS, dtype=prefix.dtype, device=dev)], dim=1)
        enc_valid = torch.cat([enc_valid, torch.ones(B, 1, dtype=torch.bool, device=dev)], dim=1)

    # ---- labels
    labels = targets.clone()
    if MASK_DOC_BOUNDARIES:
        labels = labels.masked_fill((targets == EOS).long().cumsum(dim=1) > 0, IGNORE)
    labels[short_ctx] = IGNORE

    # ---- decoder input
    K = min(int(DEC_PREFIX_TOKENS), split)
    parts = [torch.full((B, 1), BOS, dtype=batch.dtype, device=dev)]
    if K > 0:
        parts.append(prefix[:, split - K:])
    parts.append(targets[:, :-1])
    dec_in = torch.cat(parts, dim=1)
    if K > 0:
        labels = torch.cat([torch.full((B, K), IGNORE, dtype=labels.dtype, device=dev), labels], dim=1)

    return enc_ids, enc_valid, dec_in, labels, K


def seq2seq_loss(m, enc_ids, enc_valid, dec_in, labels, skip):
    """Returns (sum of token losses, number of supervised tokens)."""
    kv = None if bool(enc_valid.all()) else enc_valid       # no mask => faster attention kernels
    with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
        memory = m.encode(enc_ids, key_valid=kv)
        h, _, _ = m.decode_hidden(dec_in, memory, memory_valid=kv)
        logits = m.project(h[:, skip:, :])                  # skip the primed positions: no logits wasted on them
        tgt = labels[:, skip:]
        loss_sum = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(), tgt.reshape(-1),
            ignore_index=IGNORE, reduction="sum",
        )
    return loss_sum, (tgt != IGNORE).sum()


@torch.no_grad()
def run_self_test(m, atol=2e-3):
    was_training = m.training
    tf32 = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False           # exact fp32 so the comparison is meaningful
    m.eval()
    dev = next(m.parameters()).device
    g = torch.Generator(device="cpu").manual_seed(0)
    V = m.vocab_size
    rnd = lambda *s: torch.randint(4, V, s, generator=g).to(dev)

    B, S, T = 3, 40, 12
    enc = rnd(B, S)
    lens = torch.tensor([40, 31, 22], device=dev)
    valid = torch.arange(S, device=dev)[None, :] < lens[:, None]
    mem = m.encode(enc, key_valid=valid)

    # (a) padding invariance
    for b in range(B):
        L = int(lens[b])
        err = (mem[b, :L] - m.encode(enc[b:b + 1, :L])[0]).abs().max().item()
        assert err < atol, f"(a) encoder padding changes the output: {err:.2e}"

    # (b) cached incremental decoding == full teacher-forced decoding (incl. multi-token prefill)
    dec = rnd(B, T)
    h_full, _, _ = m.decode_hidden(dec, mem, memory_valid=valid)
    h0, sc, cc = m.decode_hidden(dec[:, :5], mem, memory_valid=valid, use_cache=True)
    outs = [h0]
    for t in range(5, T):
        ht, sc, cc = m.decode_hidden(
            dec[:, t:t + 1], mem, memory_valid=valid,
            past_self_key_values=sc, cross_key_values=cc, use_cache=True,
        )
        outs.append(ht)
    err = (h_full - torch.cat(outs, 1)).abs().max().item()
    assert err < atol, f"(b) KV-cache decoding differs from full decoding: {err:.2e}"

    # (b2) cache + several new tokens at once (causal mask with offset)
    h0, sc, cc = m.decode_hidden(dec[:, :5], mem, memory_valid=valid, use_cache=True)
    h1, _, _ = m.decode_hidden(dec[:, 5:], mem, memory_valid=valid,
                               past_self_key_values=sc, cross_key_values=cc, use_cache=True)
    err = (h_full - torch.cat([h0, h1], 1)).abs().max().item()
    assert err < atol, f"(b2) chunked cached decoding differs: {err:.2e}"

    # (c) causality
    dec2 = dec.clone(); dec2[:, -1] = 4 + (dec2[:, -1] + 1) % (V - 4)
    h2, _, _ = m.decode_hidden(dec2, mem, memory_valid=valid)
    assert (h_full[:, :-1] - h2[:, :-1]).abs().max().item() < atol, "(c) decoder is not causal"

    # (d) decoder depends on encoder, and ignores masked-out encoder keys
    mem_other = m.encode(rnd(B, S), key_valid=valid)
    h3, _, _ = m.decode_hidden(dec, mem_other, memory_valid=valid)
    assert (h_full - h3).abs().max().item() > 1e-6, "(d) decoder ignores the encoder"
    enc_junk = enc.clone(); enc_junk[1, 31:] = rnd(S - 31)
    mem_junk = m.encode(enc_junk, key_valid=valid)
    h4, _, _ = m.decode_hidden(dec, mem_junk, memory_valid=valid)
    assert (h_full[1] - h4[1]).abs().max().item() < atol, "(d) masked encoder keys leak into the decoder"

    # (e) batch builder alignment
    N, split = 300, 120
    batch = rnd(4, N + 1)
    EOS = tokenizer.eos_token_id
    batch[0, 50] = EOS      # older document inside the prompt
    batch[1, 200] = EOS     # next document starts inside the continuation
    batch[2, 119] = EOS     # prompt ends exactly at a boundary -> no usable context
    enc_ids, enc_valid, dec_in, labels, K = make_seq2seq_batch(batch.to(dev), split)
    n_cont = N + 1 - split
    assert dec_in.shape == labels.shape == (4, K + n_cont), (dec_in.shape, labels.shape)
    assert enc_ids.size(1) == split + int(ENC_END_MARKER) == enc_valid.size(1)
    if MASK_DOC_BOUNDARIES:
        assert not enc_valid[0, :51].any() and enc_valid[0, 51:].all(), "(e) encoder doc mask wrong"
        assert (labels[1, K + 80:] == IGNORE).all() and labels[1, K + 79] == batch[1, split + 79], "(e) label cut wrong"
        assert (labels[2] == IGNORE).all(), "(e) no-context row must be ignored"
        assert (labels[3, K:] == batch[3, split:]).all(), "(e) clean row labels wrong"
    assert (labels[:, :K] == IGNORE).all()
    # dec_in[K] (last primed token) must predict targets[0]: i.e. dec_in[K] == prefix[-1], label[K] == targets[0]
    assert (dec_in[:, 0] == tokenizer.bos_token_id).all()
    if K > 0:
        assert (dec_in[3, 1:K + 1] == batch[3, split - K:split].to(dev)).all()
    assert (dec_in[3, K + 1:] == batch[3, split:N].to(dev)).all(), "(e) teacher-forcing shift wrong"
    # decoder budget in the worst case
    worst = int(DEC_PREFIX_TOKENS) + SEQ_LEN + 1 - PROMPT_SPLIT_MIN
    assert worst <= m.max_decoder_positions, f"decoder positions too small: need {worst}"
    assert PROMPT_SPLIT_MAX + 1 <= m.max_encoder_positions

    torch.backends.cuda.matmul.allow_tf32 = tf32
    m.train(was_training)
    print("SELF-TEST PASSED: padding, KV-cache, causality, cross-attention, label alignment all OK.")

run_self_test(model)


SELF-TEST PASSED: padding, KV-cache, causality, cross-attention, label alignment all OK.


In [13]:
# Cell 10 — Run-mode setup
# Start with "pilot". After the pilot:
#   1) stop Cell 11,
#   2) change RUN_MODE to "full",
#   3) rerun this cell and Cell 11  (v2: Cell 11 builds fresh weights itself - no need to rerun Cell 9).

RUN_MODE = "full"   # change to "full" for the real 400M run

if RUN_MODE == "pilot":
    RUN_NAME = "encdec_pilot"
    TRAIN_BLOCK_LIMIT = 5_000_000 // SEQ_LEN   # ~5M raw corpus tokens
elif RUN_MODE == "full":
    RUN_NAME = "encdec_full"
    TRAIN_BLOCK_LIMIT = None
else:
    raise ValueError("RUN_MODE must be 'pilot' or 'full'")

CKPT_DIR = ROOT / "checkpoints" / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best.pt"
FINAL_CKPT = CKPT_DIR / "final.pt"

OUTSIDE_LATEST_CKPT = Path(f"/kaggle/working/{RUN_NAME}_latest.pt")
OUTSIDE_BEST_CKPT = Path(f"/kaggle/working/{RUN_NAME}_best.pt")

print("RUN_MODE:", RUN_MODE)
print("RUN_NAME:", RUN_NAME)
print("TRAIN_BLOCK_LIMIT:", TRAIN_BLOCK_LIMIT)
if TRAIN_BLOCK_LIMIT is not None:
    print("Approx pilot tokens:", TRAIN_BLOCK_LIMIT * SEQ_LEN)
print("Checkpoint dir:", CKPT_DIR)
print("Latest copy:", OUTSIDE_LATEST_CKPT)
print("Best copy:", OUTSIDE_BEST_CKPT)


RUN_MODE: full
RUN_NAME: encdec_full
TRAIN_BLOCK_LIMIT: None
Checkpoint dir: /kaggle/working/cs781_encdec_80m/checkpoints/encdec_full
Latest copy: /kaggle/working/encdec_full_latest.pt
Best copy: /kaggle/working/encdec_full_best.pt


In [14]:
# Cell 11 — Encoder–decoder training loop: resumable, validation-selected, 400M full-run cap
# v2: builds FRESH weights itself (then resumes from latest.pt if present), uses make_seq2seq_batch / seq2seq_loss from Cell 9b,
#     ignores cross-document labels, token-weighted validation loss over a fixed set of prompt lengths.
def get_block(mm, block_index):
    start = int(block_index) * BLOCK_TOKENS
    arr = np.asarray(mm[start:start + BLOCK_TOKENS], dtype=np.int64)
    return torch.from_numpy(arr)

def get_batch_from_indices(mm, indices):
    return torch.stack([get_block(mm, i) for i in indices], dim=0).to(
        DEVICE, non_blocking=True
    )

_val_bounds, _cum = [], 0
for _k, _s in meta["sources"].items():
    _val_bounds.append((_k, _cum, _cum + int(_s["val_tokens"])))
    _cum += int(_s["val_tokens"])

def val_source_of_block(idx):
    start = int(idx) * BLOCK_TOKENS
    for k, lo, hi in _val_bounds:
        if lo <= start < hi:
            return k
    return "other"

LAST_VAL_DETAIL = {}

@torch.no_grad()
def evaluate_val(model, max_blocks=VAL_EVAL_BLOCKS):
    """Token-weighted continuation loss. Prompt lengths come from a fixed RNG, so every call is comparable."""
    model.eval()
    n = min(N_VAL_BLOCKS, max_blocks)
    if n <= 0:
        return float("inf")

    ids = np.linspace(0, N_VAL_BLOCKS - 1, n, dtype=np.int64)
    rng = random.Random(SEED)            # separate RNG: does not disturb the training RNG stream
    tot_loss, tot_tok, by_src = 0.0, 0, {}

    for idx in ids:
        batch = get_batch_from_indices(val_mm, [idx])
        split = rng.randint(PROMPT_SPLIT_MIN, PROMPT_SPLIT_MAX)
        ls, nt = seq2seq_loss(model, *make_seq2seq_batch(batch, split))
        ls, nt = float(ls.item()), int(nt.item())
        tot_loss += ls
        tot_tok += nt
        a = by_src.setdefault(val_source_of_block(idx), [0.0, 0])
        a[0] += ls
        a[1] += nt

    LAST_VAL_DETAIL.clear()
    LAST_VAL_DETAIL["val_cont_by_source"] = {k: v[0] / max(1, v[1]) for k, v in by_src.items()}

    model.train()
    return tot_loss / max(1, tot_tok)


# Always start from freshly initialised weights (a checkpoint, if any, is loaded further below).
model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Fresh model built: {n_params/1e6:.2f}M parameters")

# AdamW: weight decay only on 2-D weight matrices (not on the tied embedding, biases or LayerNorm gains).
decay_params = [
    p for n_, p in model.named_parameters()
    if p.requires_grad and p.ndim >= 2 and "token_emb" not in n_
]
nodecay_params = [
    p for n_, p in model.named_parameters()
    if p.requires_grad and not (p.ndim >= 2 and "token_emb" not in n_)
]
param_groups = [
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": nodecay_params, "weight_decay": 0.0},
]

optim_kwargs = dict(
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
)
try:
    optimizer = torch.optim.AdamW(
        param_groups,
        fused=(DEVICE.type == "cuda"),
        **optim_kwargs,
    )
    print("AdamW fused:", DEVICE.type == "cuda")
except Exception:
    optimizer = torch.optim.AdamW(param_groups, **optim_kwargs)
    print("Using standard AdamW.")

# If compatible reused binaries happen to contain >400M tokens, full mode still caps at 400M.
AUTO_FULL_BLOCK_LIMIT = max(1, TARGET_TOTAL_TOKENS // SEQ_LEN)

N_TRAIN_USED = (
    min(N_TRAIN_BLOCKS, AUTO_FULL_BLOCK_LIMIT)
    if TRAIN_BLOCK_LIMIT is None
    else min(N_TRAIN_BLOCKS, int(TRAIN_BLOCK_LIMIT))
)

print(
    f"Training on {N_TRAIN_USED:,} of {N_TRAIN_BLOCKS:,} blocks "
    f"(~{N_TRAIN_USED * SEQ_LEN / 1e6:.0f}M raw tokens) | run={RUN_NAME}"
)

micro_steps_total = math.ceil(N_TRAIN_USED / MICRO_BATCH_SIZE)
opt_steps_total = math.ceil(micro_steps_total / GRAD_ACCUM_STEPS)
warmup_steps = max(1, int(opt_steps_total * WARMUP_RATIO))

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-6, step / max(1, warmup_steps))

    if LR_SCHEDULE == "cosine":
        progress = (step - warmup_steps) / max(1, opt_steps_total - warmup_steps)
        cosine = 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
        return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * cosine

    decay_start = int(opt_steps_total * (1.0 - WSD_DECAY_FRAC))
    if step < decay_start:
        return 1.0
    prog = (step - decay_start) / max(1, opt_steps_total - decay_start)
    return 1.0 + (MIN_LR_RATIO - 1.0) * min(1.0, prog)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

perm_gen = torch.Generator()
perm_gen.manual_seed(SEED + 777)
block_perm = torch.randperm(N_TRAIN_BLOCKS, generator=perm_gen).numpy()

cursor = 0
global_step = 0
best_val = float("inf")

def try_restore_from_hf():
    if LATEST_CKPT.exists() or not HF_CHECKPOINT_REPO:
        return
    try:
        p = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO,
            filename=f"{RUN_NAME}/latest.pt",
            repo_type="model",
            token=HF_TOKEN,
        )
        shutil.copy2(p, LATEST_CKPT)
        print("Restored latest.pt from HF checkpoint repo.")
    except Exception as e:
        print("No restorable checkpoint on HF (starting fresh):", repr(e)[:160])

try_restore_from_hf()

if LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    cursor = int(ckpt["cursor"])
    global_step = int(ckpt["global_step"])
    best_val = float(ckpt["best_val"])
    if "python_random_state" in ckpt:
        random.setstate(ckpt["python_random_state"])
    print(
        f"RESUME: cursor={cursor:,}/{N_TRAIN_USED:,} | "
        f"step={global_step:,} | best_val={best_val:.4f}"
    )

def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_copy(src, dst):
    src, dst = Path(src), Path(dst)
    tmp = Path(str(dst) + ".tmp")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def model_config():
    return {
        "architecture": "encoder_decoder_transformer",
        "vocab_size": len(tokenizer),
        "d_model": D_MODEL,
        "n_encoder_layers": N_ENCODER_LAYERS,
        "n_decoder_layers": N_DECODER_LAYERS,
        "n_heads": N_HEADS,
        "ffn_dim": FFN_DIM,
        "dropout": DROPOUT,
        "max_encoder_positions": MAX_ENCODER_POSITIONS,
        "max_decoder_positions": MAX_DECODER_POSITIONS,
        "seq_len": SEQ_LEN,
        "prompt_p10_tokens": PROMPT_P10_TOKENS,
        "prompt_p95_tokens": PROMPT_P95_TOKENS,
        "expected_cont_tokens": EXPECTED_CONT_TOKENS,
        "prompt_split_min": PROMPT_SPLIT_MIN,
        "prompt_split_max": PROMPT_SPLIT_MAX,
    }

def checkpoint_object():
    return {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "cursor": cursor,
        "global_step": global_step,
        "best_val": best_val,
        "python_random_state": random.getstate(),
        "config": model_config(),
    }

_hf_upload_count = 0
def maybe_upload_latest_to_hf():
    global _hf_upload_count
    if not HF_CHECKPOINT_REPO:
        return
    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_CHECKPOINT_REPO, repo_type="model", private=True, exist_ok=True)
        api.upload_file(
            path_or_fileobj=str(LATEST_CKPT),
            path_in_repo=f"{RUN_NAME}/latest.pt",
            repo_id=HF_CHECKPOINT_REPO,
            repo_type="model",
            commit_message=f"{RUN_NAME} checkpoint step {global_step}",
        )
        _hf_upload_count += 1
        print(f"Uploaded {RUN_NAME}/latest.pt to private HF checkpoint repo.")
    except Exception as e:
        print("HF checkpoint upload FAILED (training continues):", repr(e)[:200])

LOG_JSONL = CKPT_DIR / "train_log.jsonl"

def append_log(rec):
    rec = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), **rec}
    with open(LOG_JSONL, "a") as f:
        f.write(json.dumps(rec) + "\n")

def save_latest_and_expose():
    atomic_torch_save(checkpoint_object(), LATEST_CKPT)
    atomic_copy(LATEST_CKPT, OUTSIDE_LATEST_CKPT)
    print("Saved latest checkpoint + refreshed:", OUTSIDE_LATEST_CKPT)

run_config = {
    **model_config(),
    "run_name": RUN_NAME,
    "n_params": n_params,
    "n_train_blocks_used": int(N_TRAIN_USED),
    "opt_steps_total": int(opt_steps_total),
    "micro_batch_size": MICRO_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED,
}
json.dump(run_config, open(CKPT_DIR / "run_config.json", "w"), indent=2)

model.train()
optimizer.zero_grad(set_to_none=True)

running_loss = 0.0
running_updates = 0
train_start = time.time()
last_report_time = train_start
last_report_cursor = cursor

while cursor < N_TRAIN_USED:
    accum_loss = 0.0
    accum_count = 0

    for _ in range(GRAD_ACCUM_STEPS):
        if cursor >= N_TRAIN_USED:
            break

        inds = block_perm[cursor:min(cursor + MICRO_BATCH_SIZE, N_TRAIN_USED)]
        if len(inds) == 0:
            break

        batch = get_batch_from_indices(train_mm, inds)
        cursor += len(inds)

        # Random prompt-like split makes the encoder learn realistic Phase-1 prompt lengths.
        split = random.randint(PROMPT_SPLIT_MIN, PROMPT_SPLIT_MAX)
        loss_sum, n_tok = seq2seq_loss(model, *make_seq2seq_batch(batch, split))
        if int(n_tok.item()) == 0:
            continue                                   # every label was masked (extremely rare)

        loss = loss_sum / n_tok
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        accum_loss += float(loss.item())
        accum_count += 1

    if accum_count == 0:
        if cursor >= N_TRAIN_USED:
            break
        optimizer.zero_grad(set_to_none=True)
        continue

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

    global_step += 1
    running_loss += accum_loss / accum_count
    running_updates += 1

    if global_step % LOG_EVERY == 0:
        now = time.time()
        blocks_done = cursor - last_report_cursor
        raw_tokens_done = blocks_done * SEQ_LEN
        tok_s = raw_tokens_done / max(1e-6, now - last_report_time)
        remaining_tokens = max(0, N_TRAIN_USED - cursor) * SEQ_LEN
        eta_h = remaining_tokens / max(tok_s, 1e-9) / 3600

        avg = running_loss / max(1, running_updates)
        lr = optimizer.param_groups[0]["lr"]

        print(
            f"step={global_step:,}/{opt_steps_total:,} | "
            f"blocks={cursor:,}/{N_TRAIN_USED:,} | "
            f"processed≈{cursor * SEQ_LEN / 1e6:.1f}M raw tok | "
            f"loss={avg:.4f} | lr={lr:.2e} | "
            f"{tok_s:,.0f} raw tok/s | ETA≈{eta_h:.1f}h",
            flush=True,
        )
        append_log({
            "type":"train", "step":global_step,
            "tokens":int(cursor * SEQ_LEN),
            "loss":avg, "lr":lr, "tok_s":tok_s,
        })

        running_loss = 0.0
        running_updates = 0
        last_report_time = now
        last_report_cursor = cursor

    if global_step % VAL_EVERY == 0 or cursor >= N_TRAIN_USED:
        val_loss = evaluate_val(model)
        ppl = math.exp(min(20, val_loss))
        print(
            f"VALIDATION step={global_step:,}: "
            f"continuation_loss={val_loss:.4f} ppl={ppl:.2f}"
        )
        print(
            "  per-source continuation val loss:",
            {k: round(v,4) for k,v in LAST_VAL_DETAIL["val_cont_by_source"].items()}
        )
        append_log({
            "type":"val", "step":global_step,
            "tokens":int(cursor * SEQ_LEN),
            "val_cont":val_loss, "val_ppl":ppl,
            **LAST_VAL_DETAIL,
        })

        if val_loss < best_val:
            best_val = val_loss
            best_obj = {
                "model": model.state_dict(),
                "global_step": global_step,
                "val_loss": best_val,
                "config": model_config(),
            }
            atomic_torch_save(best_obj, BEST_CKPT)
            atomic_copy(BEST_CKPT, OUTSIDE_BEST_CKPT)
            print("Saved new BEST checkpoint + refreshed:", OUTSIDE_BEST_CKPT)

    if global_step % SAVE_EVERY == 0 or cursor >= N_TRAIN_USED:
        save_latest_and_expose()

    if HF_CHECKPOINT_REPO and (
        global_step % HF_UPLOAD_EVERY == 0 or cursor >= N_TRAIN_USED
    ):
        maybe_upload_latest_to_hf()

atomic_torch_save(
    {
        "model": model.state_dict(),
        "global_step": global_step,
        "val_loss": best_val,
        "config": model_config(),
    },
    FINAL_CKPT,
)

print("Training complete.")
print("Final checkpoint:", FINAL_CKPT)
print("Best validation loss:", best_val)
print("Latest downloadable copy:", OUTSIDE_LATEST_CKPT)
print("Best downloadable copy:", OUTSIDE_BEST_CKPT)
print("Total elapsed hours:", (time.time() - train_start) / 3600)


Fresh model built: 78.79M parameters
AdamW fused: True
Training on 260,247 of 260,247 blocks (~400M raw tokens) | run=encdec_full
step=50/16,266 | blocks=800/260,247 | processed≈1.2M raw tok | loss=7.3459 | lr=4.62e-05 | 36,152 raw tok/s | ETA≈3.1h
step=100/16,266 | blocks=1,600/260,247 | processed≈2.5M raw tok | loss=5.1096 | lr=9.23e-05 | 35,745 raw tok/s | ETA≈3.1h
step=150/16,266 | blocks=2,400/260,247 | processed≈3.7M raw tok | loss=4.3135 | lr=1.38e-04 | 35,118 raw tok/s | ETA≈3.1h
step=200/16,266 | blocks=3,200/260,247 | processed≈4.9M raw tok | loss=4.1331 | lr=1.85e-04 | 34,474 raw tok/s | ETA≈3.2h
step=250/16,266 | blocks=4,000/260,247 | processed≈6.1M raw tok | loss=4.0530 | lr=2.31e-04 | 33,964 raw tok/s | ETA≈3.2h
step=300/16,266 | blocks=4,800/260,247 | processed≈7.4M raw tok | loss=4.0308 | lr=2.77e-04 | 33,770 raw tok/s | ETA≈3.2h
step=350/16,266 | blocks=5,600/260,247 | processed≈8.6M raw tok | loss=4.0238 | lr=3.00e-04 | 33,682 raw tok/s | ETA≈3.2h
step=400/16,266 | b

In [15]:
# Cell 12 — Reload best encoder–decoder + batched cached generation
# v2: prompt priming (last K prompt tokens go through the decoder first), <s> end marker for the encoder,
#     repetition penalty no longer punishes whitespace tokens, generation settings can be passed per call (for the sweep).
best_obj = torch.load(
    BEST_CKPT if BEST_CKPT.exists() else FINAL_CKPT,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_obj["model"])
model.eval()

def mask_specials(logits, block_eos=True):
    logits = logits.clone()
    banned = [
        tokenizer.pad_token_id,
        tokenizer.unk_token_id,
        tokenizer.bos_token_id,
    ]
    if block_eos:
        banned.append(tokenizer.eos_token_id)
    logits[:, banned] = -float("inf")
    return logits

# Whitespace-only tokens are structural (this BPE keeps every space as its own token) -> never penalise them.
WS_IDS = torch.tensor(
    [i for i, t in enumerate(tokenizer.itos)
     if t not in BPE_SPECIALS and tokenizer.symbol_bytes[t].strip() == b""],
    dtype=torch.long, device=DEVICE,
)
print("whitespace-only token ids exempt from repetition penalty:", len(WS_IDS))

def apply_rep_penalty(logits, generated, penalty):
    if penalty == 1.0 or generated is None or generated.size(1) == 0:
        return logits
    sc = logits.gather(1, generated)
    out = logits.scatter(1, generated, torch.where(sc < 0, sc * penalty, sc / penalty))
    if len(WS_IDS):
        out[:, WS_IDS] = logits[:, WS_IDS]
    return out

def sample_next(logits, temperature, top_p, top_k=None):
    top_k = GEN_TOP_K if top_k is None else top_k
    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)

    logits = logits / temperature

    if top_k > 0:
        k = min(top_k, logits.size(-1))
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))

    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        pr = torch.softmax(sl, dim=-1)
        remove = (pr.cumsum(-1) - pr) > top_p
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf"))
        logits.scatter_(1, si, sl)

    return torch.multinomial(torch.softmax(logits, -1), 1)

GEN_COUNTERS = {"total":0, "short":0, "fallback":0}

@torch.inference_mode()
def generate_batch(prompts, target_words=TARGET_GENERATION_WORDS,
                   temperature=None, top_p=None, rep_penalty=None):
    """
    Encode each legal prompt once (+ <s> end marker), prime the decoder with <s> + the last K prompt tokens,
    then decode autoregressively with a self-attention KV cache and a cross-attention KV cache.
    """
    temperature = GEN_TEMPERATURE if temperature is None else temperature
    top_p = GEN_TOP_P if top_p is None else top_p
    rep_penalty = GEN_REP_PENALTY if rep_penalty is None else rep_penalty

    B = len(prompts)
    BOS = tokenizer.bos_token_id
    toks = [tokenizer.encode(p)[-PROMPT_KEEP_TOKENS:] for p in prompts]
    lens = [len(e) for e in toks]
    S = max(lens) + (1 if ENC_END_MARKER else 0)

    enc_ids = torch.full((B, S), tokenizer.pad_token_id, dtype=torch.long, device=DEVICE)
    enc_valid = torch.zeros(B, S, dtype=torch.bool, device=DEVICE)
    for b, e in enumerate(toks):                 # right padding; positions are derived from the mask (0..n-1)
        seq = e + ([BOS] if ENC_END_MARKER else [])
        enc_ids[b, :len(seq)] = torch.tensor(seq, dtype=torch.long, device=DEVICE)
        enc_valid[b, :len(seq)] = True

    K = min(int(DEC_PREFIX_TOKENS), min(lens))
    dec_rows = [[BOS] + (e[-K:] if K > 0 else []) for e in toks]
    dec = torch.tensor(dec_rows, dtype=torch.long, device=DEVICE)      # (B, 1+K)
    start_pos = dec.size(1)

    amp = torch.amp.autocast(
        device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")
    )

    with amp:
        memory = model.encode(enc_ids, key_valid=enc_valid)
        h, self_cache, cross_cache = model.decode_hidden(
            dec, memory, memory_valid=enc_valid, use_cache=True
        )

    last_h = h[:, -1, :]
    gen = torch.empty(B, 0, dtype=torch.long, device=DEVICE)

    done = [False] * B
    done_step = [None] * B
    texts = [""] * B

    hard_cap = min(GEN_MAX_NEW_TOKENS, MAX_DECODER_POSITIONS - start_pos)

    for step in range(hard_cap):
        logits = model.project(last_h).float()
        logits = mask_specials(logits, block_eos=True)
        logits = apply_rep_penalty(logits, gen, rep_penalty)
        nt = sample_next(logits, temperature, top_p)
        gen = torch.cat([gen, nt], dim=1)

        if (step + 1) % 8 == 0 or step + 1 == hard_cap:
            for b in range(B):
                if not done[b]:
                    text = tokenizer.decode(gen[b].tolist())
                    words = text.split()
                    # the last word is only final once whitespace follows it (or one more word exists)
                    if len(words) > target_words or (len(words) == target_words and text[-1:].isspace()):
                        done[b] = True
                        done_step[b] = step + 1
                        texts[b] = " ".join(words[:target_words])
            if all(done):
                break

        pos = torch.full((B, 1), start_pos + step, dtype=torch.long, device=DEVICE)
        with amp:
            h, self_cache, cross_cache = model.decode_hidden(
                nt, memory, memory_valid=enc_valid,
                past_self_key_values=self_cache, cross_key_values=cross_cache,
                use_cache=True, position_ids=pos,
            )
        last_h = h[:, -1, :]

    for b in range(B):
        GEN_COUNTERS["total"] += 1
        if not done[b]:
            words = tokenizer.decode(gen[b].tolist()).split()
            texts[b] = " ".join(words[:target_words])
            GEN_COUNTERS["short"] += 1
        elif done_step[b] > EXPECTED_CONT_TOKENS:
            # "fallback" only means generation needed more than the conservative expected budget; not an error.
            GEN_COUNTERS["fallback"] += 1

    return texts

def generate_continuation(prompt, target_words=TARGET_GENERATION_WORDS, **kw):
    return generate_batch([prompt], target_words, **kw)[0]

def generate_many(texts, desc="gen", verbose=True, **gen_kwargs):
    order = sorted(range(len(texts)), key=lambda i: len(tokenizer.encode(texts[i])))
    out = [None] * len(texts)
    t0 = time.time()
    for s in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[s:s+GEN_BATCH_SIZE]
        for i, r in zip(idx, generate_batch([texts[i] for i in idx], **gen_kwargs)):
            out[i] = r
        if verbose:
            print(
                f"  {desc}: {min(s+GEN_BATCH_SIZE, len(order))}/{len(order)} | "
                f"{time.time()-t0:.0f}s",
                flush=True,
            )
    return out

val_gen = json.load(open(VAL_GEN_PATH, encoding="utf-8"))
GEN_COUNTERS.update(total=0, short=0, fallback=0)

t0 = time.time()
val_ids = list(val_gen.keys())
val_outs = generate_many([val_gen[k]["text"] for k in val_ids], desc="val")
local_predictions = {k: {"text":o} for k,o in zip(val_ids, val_outs)}

for k in val_ids[:3]:
    print(
        f"{k} | words={len(local_predictions[k]['text'].split())} | "
        f"{local_predictions[k]['text'][:160]!r}"
    )
print("generation counters:", GEN_COUNTERS)

json.dump(
    local_predictions,
    open(LOCAL_PRED, "w", encoding="utf-8"),
    ensure_ascii=False,
    indent=2,
)
print("Generation elapsed:", f"{time.time()-t0:.1f}s")
print("Saved:", LOCAL_PRED)


whitespace-only token ids exempt from repetition penalty: 8
  val: 16/153 | 3s
  val: 32/153 | 6s
  val: 48/153 | 9s
  val: 64/153 | 12s
  val: 80/153 | 15s
  val: 96/153 | 28s
  val: 112/153 | 31s
  val: 128/153 | 34s
  val: 144/153 | 36s
  val: 153/153 | 39s
sc_00 | words=128 | 'was a consumer and that the respondent had acted in accordance with law. It appears that on November 12, 1981 the respondent had applied to the Assistant Collec'
sc_01 | words=128 | 'court has rightly decreed the suit filed by the petitioners/plaintiff for restoration of the suit. 6. Mr. D. K. Chand, learned counsel appearing for the petitio'
sc_02 | words=128 | 'under Section 34, A.P., as stated hereinabove, I am of the view that the High Court had no jurisdiction to quash the criminal proceeding under Section गया to th'
generation counters: {'total': 153, 'short': 1, 'fallback': 0}
Generation elapsed: 39.6s
Saved: /kaggle/working/cs781_encdec_80m/val_pred.json


In [16]:
# Cell 13 — Local Phase-1 evaluator: BERTScore + ROUGE-L + BLEU-4
def lcs_length(a,b):
    if not a or not b: return 0
    if len(a)<len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a):
        masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0)
        x=m|state
        y=(state<<1)|1
        state=x & ~(x-y)
    return state.bit_count()

def rouge_l_f1(pred,ref):
    pt,rt=pred.split(),ref.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    p,r=l/len(pt),l/len(rt)
    return 2*p*r/(p+r)

def ngrams(t,n):
    return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds,refs):
    clipped=[0]*4
    totals=[0]*4
    pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,5):
            pn,rn=ngrams(p,n),ngrams(r,n)
            totals[n-1]+=sum(pn.values())
            for g,c in pn.items():
                clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/4)

truth=json.load(open(VAL_REF_PATH,encoding="utf-8"))
pred=json.load(open(LOCAL_PRED,encoding="utf-8"))

if set(truth)!=set(pred):
    raise ValueError(f"ID mismatch: missing={set(truth)-set(pred)}, extra={set(pred)-set(truth)}")

ids=list(truth)
P=[pred[i]["text"].strip() for i in ids]
R=[truth[i]["text"].strip() for i in ids]

for i,p in zip(ids,P):
    if not p:
        raise ValueError(f"empty prediction for {i}")

rl=sum(rouge_l_f1(p,r) for p,r in zip(P,R))/len(P)
b4=bleu4([p.split() for p in P],[r.split() for r in R])
out={"ROUGE-L F1":rl,"BLEU-4":b4}

bert_per_sample=None
if not FAST_EVAL_NO_BERT:
    import bert_score
    _,_,f=bert_score.score(
        P,R,
        lang="en",
        batch_size=BERTSCORE_BATCH_SIZE,
        verbose=False,
        device=str(DEVICE),
    )
    bert_per_sample=f.tolist()
    out={"BERTScore F1":float(f.mean().item()),**out}

print("examples:",len(ids))
print("mean words/pred:",sum(len(p.split()) for p in P)/len(P))
print(json.dumps({k:round(v,6) for k,v in out.items()},indent=2))
# ---- per-source breakdown (ids look like "sc_00", "hc_03", ...) -> needed for the data-mixing analysis
def _src(i): return i.split("_")[0]
print("\nPer-source results:")
for s_ in sorted(set(_src(i) for i in ids)):
    ix = [k for k,i in enumerate(ids) if _src(i)==s_]
    row = {"n":len(ix), "ROUGE-L F1": round(sum(rouge_l_f1(P[k],R[k]) for k in ix)/len(ix), 4)}
    if bert_per_sample is not None:
        row["BERTScore F1"] = round(sum(bert_per_sample[k] for k in ix)/len(ix), 4)
    print(" ", s_, row)

# ---- append this decoding setting + scores to a log (use it for the decoding sweep table in the report)
with open(ROOT/"eval_log.jsonl","a") as _f:
    _f.write(json.dumps({"time":time.strftime("%Y-%m-%d %H:%M:%S"), "run":RUN_NAME,
                         "temperature":GEN_TEMPERATURE, "top_p":GEN_TOP_P, "top_k":GEN_TOP_K,
                         "rep_penalty":GEN_REP_PENALTY, "n":len(ids),
                         **{k:float(v) for k,v in out.items()}}) + "\n")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


examples: 153
mean words/pred: 128.0
{
  "BERTScore F1": 0.819646,
  "ROUGE-L F1": 0.183109,
  "BLEU-4": 0.034743
}

Per-source results:
  acts {'n': 33, 'ROUGE-L F1': 0.1896, 'BERTScore F1': 0.8155}
  hc {'n': 40, 'ROUGE-L F1': 0.1787, 'BERTScore F1': 0.8206}
  sc {'n': 40, 'ROUGE-L F1': 0.1902, 'BERTScore F1': 0.8223}
  tr {'n': 40, 'ROUGE-L F1': 0.175, 'BERTScore F1': 0.8194}


In [17]:
# Cell 13b — Decoding sweep: cheap overlap sweep, then BERTScore on the top candidates
# Phase 1 is ranked primarily by BERTScore F1.  We therefore use ROUGE-L/BLEU
# to cheaply screen several decoding settings, then compute BERTScore for the
# best few candidates and choose the final setting by BERTScore.
#
# After this cell, copy the printed recommended values into Cell 2 before
# official generation (or set the three globals in a tiny override cell).

SWEEP = [
    dict(temperature=0.0, top_p=1.00, rep_penalty=1.00),   # greedy
    dict(temperature=0.0, top_p=1.00, rep_penalty=1.05),   # greedy + repetition control
    dict(temperature=0.3, top_p=0.95, rep_penalty=1.03),
    dict(temperature=0.5, top_p=0.95, rep_penalty=1.03),
    dict(temperature=0.7, top_p=0.95, rep_penalty=1.03),
    dict(temperature=0.8, top_p=0.95, rep_penalty=1.03),
]

SWEEP_BERT_TOP_N = 3

truth_sw = json.load(open(VAL_REF_PATH, encoding="utf-8"))
val_gen_sw = json.load(open(VAL_GEN_PATH, encoding="utf-8"))
ids_sw = list(val_gen_sw.keys())
R_sw = [truth_sw[i]["text"].strip() for i in ids_sw]

rows = []

print("Stage 1/2: generation + ROUGE-L/BLEU screening")
for j, cfg in enumerate(SWEEP):
    # Same seed for each stochastic setting -> fairer comparison.
    torch.manual_seed(SEED + 9100)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + 9100)

    t0 = time.time()
    outs = generate_many(
        [val_gen_sw[i]["text"] for i in ids_sw],
        verbose=False,
        **cfg,
    )
    P_sw = [o.strip() for o in outs]

    row = {
        "idx": j,
        **cfg,
        "ROUGE-L": sum(
            rouge_l_f1(p, r) for p, r in zip(P_sw, R_sw)
        ) / len(P_sw),
        "BLEU-4": bleu4(
            [p.split() for p in P_sw],
            [r.split() for r in R_sw],
        ),
        "sec": time.time() - t0,
        "_preds": P_sw,
    }
    rows.append(row)

    printable = {k:v for k,v in row.items() if k != "_preds"}
    print({
        k:(round(v, 5) if isinstance(v, float) else v)
        for k,v in printable.items()
    }, flush=True)

# Shortlist primarily by ROUGE-L, with BLEU as a tie-breaker.
shortlist = sorted(
    rows,
    key=lambda r:(r["ROUGE-L"], r["BLEU-4"]),
    reverse=True,
)[:min(SWEEP_BERT_TOP_N, len(rows))]

print("\nStage 2/2: BERTScore on top candidates (primary Phase-1 metric)")

import bert_score

for row in shortlist:
    _, _, f = bert_score.score(
        row["_preds"],
        R_sw,
        lang="en",
        batch_size=BERTSCORE_BATCH_SIZE,
        verbose=False,
        device=str(DEVICE),
    )
    row["BERTScore F1"] = float(f.mean().item())

    printable = {k:v for k,v in row.items() if k != "_preds"}
    print({
        k:(round(v, 6) if isinstance(v, float) else v)
        for k,v in printable.items()
    }, flush=True)

    with open(ROOT / "eval_log.jsonl", "a") as _f:
        _f.write(json.dumps({
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
            "run": RUN_NAME,
            "kind": "decoding_sweep",
            "n": len(row["_preds"]),
            **{k:v for k,v in row.items() if k != "_preds"},
        }) + "\n")

best = max(shortlist, key=lambda r:r["BERTScore F1"])

print("\n✅ Recommended decoding setting by BERTScore:")
print({
    "temperature": best["temperature"],
    "top_p": best["top_p"],
    "rep_penalty": best["rep_penalty"],
    "BERTScore F1": round(best["BERTScore F1"], 6),
    "ROUGE-L": round(best["ROUGE-L"], 6),
    "BLEU-4": round(best["BLEU-4"], 6),
})

print("\nBefore official generation, run:")
print(f"GEN_TEMPERATURE = {best['temperature']!r}")
print(f"GEN_TOP_P = {best['top_p']!r}")
print(f"GEN_REP_PENALTY = {best['rep_penalty']!r}")

# Release retained prediction lists.
for row in rows:
    row.pop("_preds", None)


Stage 1/2: generation + ROUGE-L/BLEU screening
{'idx': 0, 'temperature': 0.0, 'top_p': 1.0, 'rep_penalty': 1.0, 'ROUGE-L': 0.16845, 'BLEU-4': 0.02169, 'sec': 28.56899}
{'idx': 1, 'temperature': 0.0, 'top_p': 1.0, 'rep_penalty': 1.05, 'ROUGE-L': 0.17193, 'BLEU-4': 0.02603, 'sec': 28.43691}
{'idx': 2, 'temperature': 0.3, 'top_p': 0.95, 'rep_penalty': 1.03, 'ROUGE-L': 0.17093, 'BLEU-4': 0.02559, 'sec': 50.98937}
{'idx': 3, 'temperature': 0.5, 'top_p': 0.95, 'rep_penalty': 1.03, 'ROUGE-L': 0.17581, 'BLEU-4': 0.03005, 'sec': 37.71375}
{'idx': 4, 'temperature': 0.7, 'top_p': 0.95, 'rep_penalty': 1.03, 'ROUGE-L': 0.17887, 'BLEU-4': 0.03451, 'sec': 34.12172}
{'idx': 5, 'temperature': 0.8, 'top_p': 0.95, 'rep_penalty': 1.03, 'ROUGE-L': 0.17781, 'BLEU-4': 0.02979, 'sec': 40.53298}

Stage 2/2: BERTScore on top candidates (primary Phase-1 metric)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'idx': 4, 'temperature': 0.7, 'top_p': 0.95, 'rep_penalty': 1.03, 'ROUGE-L': 0.178871, 'BLEU-4': 0.034514, 'sec': 34.121723, 'BERTScore F1': 0.816979}


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'idx': 5, 'temperature': 0.8, 'top_p': 0.95, 'rep_penalty': 1.03, 'ROUGE-L': 0.177812, 'BLEU-4': 0.02979, 'sec': 40.532979, 'BERTScore F1': 0.816951}


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'idx': 3, 'temperature': 0.5, 'top_p': 0.95, 'rep_penalty': 1.03, 'ROUGE-L': 0.175807, 'BLEU-4': 0.030048, 'sec': 37.713752, 'BERTScore F1': 0.805698}

✅ Recommended decoding setting by BERTScore:
{'temperature': 0.7, 'top_p': 0.95, 'rep_penalty': 1.03, 'BERTScore F1': 0.816979, 'ROUGE-L': 0.178871, 'BLEU-4': 0.034514}

Before official generation, run:
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.95
GEN_REP_PENALTY = 1.03


In [18]:
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.95
GEN_REP_PENALTY = 1.03

RUN_OFFICIAL_TEST = True

print(GEN_TEMPERATURE, GEN_TOP_P, GEN_REP_PENALTY)
print("RUN_OFFICIAL_TEST =", RUN_OFFICIAL_TEST)

0.7 0.95 1.03
RUN_OFFICIAL_TEST = True


In [19]:
# Cell 14 — Official generation + automatic exact-128 repair + Codabench ZIP
# Run this small override first when you are ready:
#     RUN_OFFICIAL_TEST = True
#
# The cell is resumable through prediction_partial.json.

if RUN_OFFICIAL_TEST:
    test_path = find_test_file()
    test = {str(k):v for k,v in json.load(open(test_path, encoding="utf-8")).items()}

    fingerprint = {
        "architecture": "encoder_decoder_transformer",
        "ckpt_step": best_obj.get("global_step"),
        "temperature": GEN_TEMPERATURE,
        "top_p": GEN_TOP_P,
        "top_k": GEN_TOP_K,
        "rep_penalty": GEN_REP_PENALTY,
        "prompt_keep": PROMPT_KEEP_TOKENS,
        "max_decoder_positions": MAX_DECODER_POSITIONS,
        "dec_prefix_tokens": DEC_PREFIX_TOKENS,
        "enc_end_marker": ENC_END_MARKER,
        "generation_seed": GENERATION_SEED,
    }
    partial_path = ROOT / "prediction_partial.json"

    def atomic_json(obj, path):
        tmp = Path(str(path) + ".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False)
        os.replace(tmp, path)

    predictions = {}
    if partial_path.exists():
        saved = json.load(open(partial_path, encoding="utf-8"))
        if saved.get("fingerprint") == fingerprint:
            predictions = saved.get("predictions", {})
            print(f"Resuming official generation: {len(predictions)} prompts already done.")
        else:
            print("Ignoring stale partial file.")

    todo = [k for k in test if k not in predictions]
    todo.sort(key=lambda k: len(tokenizer.encode(test[k]["text"])))

    print(f"Official prompts: {len(test)} | to generate now: {len(todo)}")

    # A brand-new official run starts from a known sampling seed.
    # Existing completed predictions remain untouched when resuming a partial file.
    if len(predictions) == 0:
        torch.manual_seed(GENERATION_SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(GENERATION_SEED)
        print("Official generation seed:", GENERATION_SEED)

    GEN_COUNTERS.update(total=0, short=0, fallback=0)
    t0 = time.time()

    for n, s in enumerate(range(0, len(todo), GEN_BATCH_SIZE), 1):
        chunk = todo[s:s+GEN_BATCH_SIZE]
        outs = generate_batch([test[k]["text"] for k in chunk])

        for k, o in zip(chunk, outs):
            predictions[k] = {"text":o}

        if n % OFFICIAL_PARTIAL_EVERY == 0 or s + GEN_BATCH_SIZE >= len(todo):
            atomic_json(
                {"fingerprint":fingerprint, "predictions":predictions},
                partial_path,
            )

        print(
            f"{len(predictions)}/{len(test)} | elapsed={time.time()-t0:.0f}s",
            flush=True,
        )

    if set(predictions) != set(test):
        raise RuntimeError("Official test ID mismatch.")

    # Rare short generations are regenerated one-at-a-time. With an independent
    # 1536-position decoder this normally needs no special prompt shortening.
    bad_ids = [
        k for k,v in predictions.items()
        if len(v["text"].split()) != TARGET_GENERATION_WORDS
    ]

    if bad_ids:
        print("Repairing short IDs:", {
            k: len(predictions[k]["text"].split()) for k in bad_ids
        })

        for k in bad_ids:
            fixed = None

            # A few independent sampling attempts are cheap and preserve the prompt.
            # Stable per-ID seeds make a repair reproducible.
            for attempt in range(6):
                repair_seed = GENERATION_SEED + 100_000 + int(k) * 17 + attempt
                torch.manual_seed(repair_seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(repair_seed)

                text = generate_batch(
                    [test[k]["text"]],
                    target_words=TARGET_GENERATION_WORDS,
                )[0]
                if len(text.split()) == TARGET_GENERATION_WORDS:
                    fixed = text
                    print(f"  repaired {k} on attempt {attempt+1}")
                    break

            if fixed is None:
                raise RuntimeError(
                    f"Could not produce exactly 128 words for official ID {k}. "
                    "Do not submit until repaired."
                )

            predictions[k] = {"text":fixed}

    # Strict final validation before writing the ZIP.
    assert set(predictions) == set(test)
    bad = {
        k: len(v["text"].split())
        for k,v in predictions.items()
        if len(v["text"].split()) != TARGET_GENERATION_WORDS
    }
    if bad:
        raise RuntimeError(f"Wrong word counts remain: {bad}")

    json.dump(
        predictions,
        open(OFFICIAL_PRED, "w", encoding="utf-8"),
        ensure_ascii=False,
        indent=2,
    )

    with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(OFFICIAL_PRED, arcname="prediction.json")

    # Validate the actual ZIP that will be submitted.
    with zipfile.ZipFile(SUBMISSION_ZIP, "r") as z:
        names = z.namelist()
        assert names == ["prediction.json"], names
        zipped = json.loads(z.read("prediction.json").decode("utf-8"))

    zipped_bad = {
        k:len(v["text"].split())
        for k,v in zipped.items()
        if len(v["text"].split()) != TARGET_GENERATION_WORDS
    }

    assert len(zipped) == len(test) == 1000
    assert not zipped_bad, zipped_bad

    print("generation counters:", GEN_COUNTERS)
    print("Predictions:", len(zipped))
    print("Min words:", min(len(v["text"].split()) for v in zipped.values()))
    print("Max words:", max(len(v["text"].split()) for v in zipped.values()))
    print("ZIP contents:", names)
    print("prediction.json:", OFFICIAL_PRED)
    print("submission.zip:", SUBMISSION_ZIP)
    print("✅ OFFICIAL SUBMISSION PASSED ALL STRUCTURAL CHECKS")
else:
    print("RUN_OFFICIAL_TEST=False — official generation intentionally skipped.")


Official test file: Phase_1_Pretraining/Test/test.json
Official prompts: 1000 | to generate now: 1000
Official generation seed: 3260
16/1000 | elapsed=3s
32/1000 | elapsed=17s
48/1000 | elapsed=19s
64/1000 | elapsed=22s
80/1000 | elapsed=25s
96/1000 | elapsed=27s
112/1000 | elapsed=30s
128/1000 | elapsed=33s
144/1000 | elapsed=36s
160/1000 | elapsed=38s
176/1000 | elapsed=41s
192/1000 | elapsed=44s
208/1000 | elapsed=47s
224/1000 | elapsed=50s
240/1000 | elapsed=53s
256/1000 | elapsed=56s
272/1000 | elapsed=58s
288/1000 | elapsed=61s
304/1000 | elapsed=64s
320/1000 | elapsed=67s
336/1000 | elapsed=69s
352/1000 | elapsed=72s
368/1000 | elapsed=75s
384/1000 | elapsed=90s
400/1000 | elapsed=93s
416/1000 | elapsed=96s
432/1000 | elapsed=98s
448/1000 | elapsed=101s
464/1000 | elapsed=104s
480/1000 | elapsed=114s
496/1000 | elapsed=117s
512/1000 | elapsed=119s
528/1000 | elapsed=122s
544/1000 | elapsed=137s
560/1000 | elapsed=140s
576/1000 | elapsed=142s
592/1000 | elapsed=145s
608/1000 | el

In [20]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "encdec_full" in file and file.endswith(".pt"):
            print(os.path.join(root, file))

/kaggle/input/datasets/aaryan1234567890/phase1-model3-ckpts/encdec_full_latest (1).pt
/kaggle/input/datasets/aaryan1234567890/phase1-model3-ckpts/encdec_full_best (2).pt


In [21]:
import os
import torch

paths = [
    "/kaggle/input/datasets/aaryan1234567890/phase1-model3-ckpts/encdec_full_latest (1).pt",
    "/kaggle/input/datasets/aaryan1234567890/phase1-model3-ckpts/encdec_full_best (2).pt",
]

def inspect_checkpoint(path):
    print("\n" + "="*100)
    print("PATH:", path)

    assert os.path.exists(path), f"File not found: {path}"

    obj = torch.load(path, map_location="cpu", weights_only=False)

    print("✅ loads correctly")
    print("Size GB:", round(os.path.getsize(path)/(1024**3), 3))
    print("Keys:", list(obj.keys()))

    print("global_step:", obj.get("global_step"))
    print("cursor:", obj.get("cursor"))
    print("best_val:", obj.get("best_val"))
    print("val_loss:", obj.get("val_loss"))

    resume_keys = {
        "model",
        "optimizer",
        "scheduler",
        "scaler",
        "cursor",
        "global_step",
        "best_val",
        "python_random_state",
        "config",
    }

    missing = resume_keys - set(obj.keys())

    if not missing:
        print("✅ FULLY RESUMABLE TRAINING CHECKPOINT")
    else:
        print("ℹ️ MODEL / BEST CHECKPOINT")
        print("Missing resume keys:", sorted(missing))

    if "config" in obj:
        print("\nCONFIG:")
        for k, v in obj["config"].items():
            print(f"  {k}: {v}")

for p in paths:
    inspect_checkpoint(p)


PATH: /kaggle/input/datasets/aaryan1234567890/phase1-model3-ckpts/encdec_full_latest (1).pt
✅ loads correctly
Size GB: 0.881
Keys: ['model', 'optimizer', 'scheduler', 'scaler', 'cursor', 'global_step', 'best_val', 'python_random_state', 'config']
global_step: 16266
cursor: 260247
best_val: 1.885548301174893
val_loss: None
✅ FULLY RESUMABLE TRAINING CHECKPOINT

CONFIG:
  architecture: encoder_decoder_transformer
  vocab_size: 24576
  d_model: 512
  n_encoder_layers: 9
  n_decoder_layers: 9
  n_heads: 8
  ffn_dim: 2048
  dropout: 0.0
  max_encoder_positions: 2048
  max_decoder_positions: 1536
  seq_len: 1536
  prompt_p10_tokens: 1050
  prompt_p95_tokens: 1139
  expected_cont_tokens: 384
  prompt_split_min: 1050
  prompt_split_max: 1139

PATH: /kaggle/input/datasets/aaryan1234567890/phase1-model3-ckpts/encdec_full_best (2).pt
✅ loads correctly
Size GB: 0.294
Keys: ['model', 'global_step', 'val_loss', 'config']
global_step: 16266
cursor: None
best_val: None
val_loss: 1.885548301174893
ℹ️ 

## Final checks — Architecture 3 (v3 FINAL)

- tokenizer round-trip `True`, vocabulary 24,576, ~79M parameters;
- **Cell 9b prints `SELF-TEST PASSED`** (run it again after any model edit);
- preprocessing reports ≈400M train token IDs (the data cell does not enforce this — check the printed per-source totals);
- pilot: loss should fall steadily from ≈ln(24576)=10.1; check tokens/s and ETA before committing to the full run.

Easy-download checkpoint mirrors: `/kaggle/working/encdec_full_latest.pt` (resumable) and `/kaggle/working/encdec_full_best.pt` (validation-selected).
Note: v1 checkpoints are **not** compatible with v2 results (embedding scaling, priming and masking change what the weights mean) — train v2 from scratch.

- The decoding sweep now uses BERTScore for the final decoding-setting choice; this aligns model selection with the Phase-1 primary metric.
- A fresh official generation starts from a fixed sampling seed and short-output repair attempts use stable per-ID seeds.
- The training objective remains unchanged: encoder prompt → autoregressive legal continuation.
